###Step 1 : Import Libraries & API Keys

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
import gradio as gr
import json
#OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")

print(OPENAI_API_KEY[:8])

###Step 2: Set up Pushover

In [ ]:
#from dotenv import load_dotenv
#import os

#load_dotenv()

#print("Loaded:", os.getenv("OPENAI_API_KEY") is not None)

In [ ]:
#Step 2a -> set up account in browser pushover
#Step 2b-> Set up the app on your iphone/Android phone, Log into the same account
#step 2c-> In the browser create an "Application/API Token"
#Step 2d-> copy your user key and API token into the .env file
#like this but without the hastag symbols and with your own keys
#PUSHOVER_USER=####
#PUSHOVER_TOKEN=####
#SAVE CHANGES    TO THE .ENV FILES
#Run the test manually send a notification

In [ ]:

load_dotenv()

In [ ]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [ ]:
#Test Pushover
import requests

def send_notification(message: str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
#send_notification("Hello to myselft, from this AI engineering training")

###Step 3: Describe Pushover as an LLM tool

In [ ]:
send_notification_function = {
    "name": "send_notification",
    "description": "Send one notification. Call this two times if multiple notifications are needed.",
    "parameters":{
        "type": "object",
        "properties":{
            "message":{
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ["message"]
}
}

###STep 4: Add Pushover to the list of tool for the LLM

In [ ]:
tools = [{"type":"function", "function":send_notification_function}]

###Steps 2b & 3b & 4b: Create new function, describe it, add it to the list of tools

In [ ]:
import random
#simulates rolling a single six-sided die
def dice_roll():
    result = random.randint(1,6)
    return  result

#Describe fucntion for the LLM
roll_dice_function = {
      "name": "dice_roll",
      "description": "Simulates rolling a single six-sided die and returns the result. Use this when the user wants to roll a die for games, or random number generation.",
      "parameters":{
         "type": "object",
         "properties":{},
         "required": []
    }
}


#Add fucntion to list of tools of LLM
tools.append({"type":"function", "function":roll_dice_function})


###Step 5: Calling the tool from an LLM

In [ ]:
def handle_tool_call(tool_calls):
    tool_results = []

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        #print(f" Calling function {function_name}")

        #Route to the appropriate function based on function_name
        if function_name == "send_notification":
             #tool_call = tool_calls[0] #assuming just one tool call
            send_notification(args["message"])
            content = f"Notification sent: {args["message"]}"
        elif function_name == "dice_roll":
            content = f"Rolled: {dice_roll()}"
         #elif function_name == "insert_function_name_3":
        #    content = insert_function_name_3(args["messag"])

        else:
            content = f"Unknown function: {function_name}"




           #Actually send the notification, i.e, call the tool
        
            #print(f"Sent notification: {args['message']}")

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
    }

    
    
    return tool_results   #what to add to our "context"(about tool call), a dictionary.

In [ ]:
import json
import random

def handle_tool_call(tool_calls):
    results = []

    for tool_call in tool_calls:   # ✅ loop through ALL calls
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        if function_name == "roll_dice":
            dice1 = random.randint(1, 6)
            dice2 = random.randint(1, 6)
            result = f"Dice rolled: {dice1}, {dice2}"

        elif function_name == "send_notification":
            msg = args.get("message", "")
            result = f"Notification sent: {msg}"

        else:
            result = "Unknown tool"

        results.append({
            "role": "tool",
            "tool_call_id": tool_call.id,   # ✅ MUST match exactly
            "content": result
        })
        
    return results

In [ ]:
client = OpenAI()
messages=[
      {"role": "user",
      "content": "Please do 2 things:\
                 1) I'd like to roll 2 dice, amd\
                 2) Send me a notification saying hey there"}
    ]
     

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
    tools=tools
)

message = response.choices[0].message

#check if model wants to call a tool
     
while message.tool_calls:
    from pprint import pprint
    tool_result = handle_tool_call(message.tool_calls)
    messages.append(message)
    messages.extend(tool_result)

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools
    )
    message = response.choices[0].message

    #maybe consider adding protection from infinite consecutive tool calling

print(message.content)

In [ ]:
client = OpenAI()
messages = [
    {"role": "user",
     "content": "Please do 2 things: "
                "1) I'd like to roll 2 dice, and "
                "2) Send me a notification saying hey there"}
]

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
    tools=tools
)
message = response.choices[0].message

while message.tool_calls:
    tool_results = []
    for tool_call in message.tool_calls:       # ✅ loop over ALL tool calls
        output = handle_tool_call([tool_call])  # call your handler for each
        tool_results.extend(output)

    messages.append(message)                   # assistant message first
    messages.extend(tool_results)              # then ALL tool results

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools
    )
    message = response.choices[0].message

print(message.content)